# Mortgage Default Prediction: 5-Model Classification Comparison

**Academic project.** Predict whether a mortgage holder will default (`Default on Mortgage?`, YES/NO) from age, marital status, annual income, mortgage amount, and total amount paid. Two KNN models and three decision trees are compared on a held-out test set, and the winner scores 20 new applicants.

> **Dataset not included (course-provided).** `mortgagedefault.csv` (300 records) and `mortgage_prediction.csv` (20 applicants) were supplied for a university course and are not redistributed. The notebook is published **without outputs**; numbers quoted in the markdown come from the original course run. To run it, place both files in `data/`.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, accuracy_score, f1_score

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
dataset_path = str(ROOT / 'data')

## Phase 1: Load data and split 80/20

`random_state=1` gives 240 training and 60 test records.

In [ ]:
mortgage = pd.read_csv(dataset_path + '/mortgagedefault.csv')

X_tree = ['Age', 'Marital Status', 'Annual Income ($)', 'Mortgage Amount ($)', 'Total Amount Paid ($)']
y_tree = 'Default on Mortgage?'

X = mortgage[X_tree]
y = mortgage[y_tree]

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
print(f'Train: {len(x_train)} | Test: {len(x_test)}')
print(y.value_counts())

## Phase 2: Train five models

- KNN with k = 5 and k = 10
- Entropy decision trees with `min_impurity_decrease=0.03`: a baseline, one with `min_samples_split=70`, and one with `min_samples_leaf=50`

In [ ]:
knn_5 = KNeighborsClassifier(n_neighbors=5).fit(x_train, y_train)
knn_10 = KNeighborsClassifier(n_neighbors=10).fit(x_train, y_train)

mortgage_tree = DecisionTreeClassifier(criterion='entropy', min_impurity_decrease=0.03).fit(x_train, y_train)
mortgage_tree_minsamplesplit = DecisionTreeClassifier(
    criterion='entropy', min_samples_split=70, min_impurity_decrease=0.03).fit(x_train, y_train)
mortgage_tree_minsampleleaf = DecisionTreeClassifier(
    criterion='entropy', min_samples_leaf=50, min_impurity_decrease=0.03).fit(x_train, y_train)

## Phase 3: Evaluation

In [ ]:
models = {
    'knn_5': knn_5,
    'knn_10': knn_10,
    'mortgage_tree': mortgage_tree,
    'mortgage_tree_minsamplesplit': mortgage_tree_minsamplesplit,
    'mortgage_tree_minsampleleaf': mortgage_tree_minsampleleaf,
}

rows = []
for name, model in models.items():
    y_pred = model.predict(x_test)
    print(f'===== {name} =====')
    print(classification_report(y_test, y_pred, target_names=model.classes_))
    complexity = model.n_neighbors if hasattr(model, 'n_neighbors') else model.get_n_leaves()
    rows.append({'Model Name': name,
                 'F1-score (weighted avg)': round(f1_score(y_test, y_pred, average='weighted'), 4),
                 'Accuracy': round(accuracy_score(y_test, y_pred), 4),
                 'Complexity (k or leaves)': complexity})

comparison_table = pd.DataFrame(rows)
comparison_table

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_tree(mortgage_tree, feature_names=X_tree, class_names=list(mortgage_tree.classes_), filled=True, ax=ax)
ax.set_title('Baseline decision tree (entropy, min_impurity_decrease=0.03)')
plt.show()

**Original course run:** `knn_5`, `knn_10`, and the baseline tree all reached 77% accuracy; `knn_5` had the best weighted F1 (0.7656 vs 0.7640 for the baseline tree). `knn_10` caught 91% of NO cases but only 59% of defaults (YES), while `knn_5` recalled about 70% of defaults. The `min_samples_leaf=50` tree collapsed to 4 leaves, 61.7% accuracy, and 37% YES recall. `knn_5` was selected.

## Phase 4: Score 20 new applicants with knn_5

In [ ]:
mortgage_prediction = pd.read_csv(dataset_path + '/mortgage_prediction.csv')
mortgage_prediction['Default on Mortgage?'] = knn_5.predict(mortgage_prediction[X_tree])
print(mortgage_prediction['Default on Mortgage?'].value_counts())
mortgage_prediction

**Original course run:** `knn_5` flagged 8 of 20 applicants (40%) as likely to default, skewing toward younger applicants (24 to 31) with lower income and high mortgage-to-income ratios.

## Limitations and next steps

- Features are not scaled, which matters for KNN because income and mortgage amounts dominate the distance. Next step: `StandardScaler` in a `Pipeline`.
- With 60 test records, a few predictions move accuracy by several points; cross-validation would give more stable estimates.
- For credit risk, recall on defaults (YES) matters more than overall accuracy, so threshold tuning or class weights are worth testing.